# Mathematics and Statistics for Data Analysis – Homework 4

**Student:** Yusra Qayyum  
**Course:** Mathematics and Statistics for Data Analysis  
**Assignment:** Homework 4 — Theory + Computational Applications

This notebook contains:

- **Part 1 – Geometric Foundations and Best Approximation**  
  Conceptual questions on vector spaces, norms, inner products, Hilbert spaces,
  best approximation, Gram matrices, dual bases, and stability.

- **Part 2 – EVD, SVD, and Applications**  
  Theory questions on eigenvalue and singular value decompositions, pseudoinverse,
  instability, regularization, TLS, PCA, and the Gram operator / Riesz bases.

- **Part 3 – Computational Applications (Coding)**  
  Code demonstrations for:
  - Q13: Gram Matrix, Least Squares, and Truncated SVD
  - Q14: When Does Total Least Squares Help?
  - Q15: PCA and Low-Rank Approximation (digit image)
  - Q18: Regression with Explicit and Kernel Features

The code in this notebook **reuses the modular `.py` files in `src/`**, and
visualizations/logs are saved under `outputs/figures/` and `outputs/logs/`.


In [8]:
%cd ..

/home/saad-alam/Documents/assignments/Yusra's_stats/homework4


/home/saad-alam/Documents/assignments/Yusra's_stats/homework4/venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [9]:
import numpy as np
import matplotlib.pyplot as plt
# Import the computational modules implemented in src/
from src.monomial_ls_tsvd import run_monomial_experiment
from src.tls_experiments import run_tls_experiments
from src.pca_analysis import run_pca_steps
from src.regression_kernels import run_regression_comparison

## Part 1 – Q1: Conceptual Summary – The “Story” of Geometric Spaces

We start with a **vector space** \(S\) over a field \(\mathbb{F}\) (typically \(\mathbb{R}\) or \(\mathbb{C}\)).
A vector space has only the **algebraic** structure: we can add vectors and
scale them, and these operations satisfy the usual axioms (associativity,
commutativity of addition, distributivity, existence of zero and additive
inverses, etc.). At this level, there is **no notion of length, angle, or
distance**.

A **normed space** \((S, \|\cdot\|)\) adds a **norm**:
\[
\|x\| \ge 0,\quad \|x\|=0 \iff x=0,\quad \|\alpha x\| = |\alpha|\|x\|,\quad
\|x+y\| \le \|x\| + \|y\|.
\]
The norm gives us a notion of **length** and hence distance
\(d(x,y) = \|x-y\|\). This allows us to talk about convergence of sequences and
continuity of linear maps, but we still do not have a notion of angle or
orthogonality.

An **inner product space** \((S, \langle\cdot,\cdot\rangle)\) adds an
**inner product**, a bilinear (or sesquilinear) form satisfying:
\[
\langle x,y\rangle = \overline{\langle y,x\rangle},\quad
\langle x,x\rangle \ge 0,\quad
\langle x,x\rangle = 0 \iff x=0.
\]
The induced norm \(\|x\| = \sqrt{\langle x,x\rangle}\) now comes
from geometry; we can define **angles**, **orthogonality**, and **projections**:
\(\langle x,y\rangle=0\) means “orthogonal”. This allows us to formulate and
solve the **best approximation problem** via orthogonal projection onto a
subspace.

A **Hilbert space** is an inner product space that is also **complete** with
respect to the induced norm: every Cauchy sequence in the space converges to a
limit in the space. Completeness is crucial because many useful bases and
expansions (e.g. Fourier series) are infinite. Without completeness, Cauchy
sequences of partial sums might converge to limits outside the space, and the
best approximation may **fail to exist**. In a Hilbert space, we are guaranteed
that orthogonal projections onto **closed** subspaces exist and give the unique
best approximation.

In summary: we move from purely algebraic structure (vector spaces) to geometric
structure (norms and inner products), and then to analytic robustness
(completeness in Hilbert spaces). This hierarchy is precisely what allows us to
define and solve the best approximation problem:
given \(f\) and a closed subspace \(T\), the optimal approximation
\(\hat f \in T\) is the orthogonal projection characterized by
\(\langle f-\hat f, v\rangle = 0\) for all \(v\in T\).

## Part 1 – Q2: Testing the Subspace Axioms

We use the characterization: a non-empty subset \(T \subset S\) is a **subspace**
iff for all \(x,y \in T\) and all scalars \(a,b\), the linear combination
\(ax + by \in T\). Equivalently, \(T\) is closed under addition and scalar
multiplication and contains the zero vector.

### (a) \(S = \mathbb{R}^3\), \(T = \{ x \in \mathbb{R}^3 : x_1 - 2x_2 + x_3 = 1\}\)

Take \(x = (1,0,0)\). Then
\[
x_1 - 2x_2 + x_3 = 1 - 0 + 0 = 1,
\]
so \(x \in T\). Now check whether \(0 \in T\):
\[
(0)_1 - 2(0)_2 + (0)_3 = 0 \neq 1,
\]
so the zero vector is **not** in \(T\). Therefore \(T\) cannot be a subspace.
(Geometrically, \(T\) is an affine plane not passing through the origin.)

**Conclusion:** \(T\) is **not** a subspace of \(\mathbb{R}^3\).

---

### (b) \(S = C[a,b]\), \(T = \{ f \in S : f(t)\ge 0 \text{ for all } t\in[a,b]\}\)

If \(f \in T\), then \(f(t) \ge 0\) for all \(t\).
Consider scalar multiplication by a negative scalar, say \(\alpha = -1\). Then
\((\alpha f)(t) = -f(t) \le 0\), and unless \(f \equiv 0\), we have
\((\alpha f)(t) < 0\) somewhere. Thus \(\alpha f \notin T\) in general.

This shows **closure under scalar multiplication fails**. Therefore \(T\) is not
a subspace.

**Conclusion:** \(T\) is **not** a subspace of \(C[a,b]\).

---

### (c) \(S = L^2[a,b]\), \(T = \{ f \in S : \int_a^b f(t)\,dt = 0 \}\)

Take any \(f,g \in T\) and scalars \(a,b\). Then
\[
\int_a^b (af(t) + bg(t))\, dt
= a \int_a^b f(t)\,dt + b \int_a^b g(t)\,dt
= a \cdot 0 + b \cdot 0 = 0.
\]
So \(af + bg \in T\). Also, the zero function clearly satisfies
\(\int_a^b 0\,dt = 0\), so \(0 \in T\).

Thus \(T\) is closed under addition and scalar multiplication and contains the
zero element.

**Conclusion:** \(T\) **is** a subspace of \(L^2[a,b]\).

## Part 1 – Q3: Proving Norms and Non-Norms

We recall the three axioms of a norm \(\|\cdot\|\):

1. **Non-negativity & definiteness:** \(\|x\|\ge 0\) and \(\|x\|=0\) iff \(x=0\).
2. **Homogeneity:** \(\|\alpha x\| = |\alpha| \|x\|\) for all scalars \(\alpha\).
3. **Triangle inequality:** \(\|x+y\| \le \|x\| + \|y\|\).

---

### (a) The induced norm \(\|x\| := \sqrt{\langle x,x\rangle}\)

Assume \(\langle\cdot,\cdot\rangle\) is a valid inner product.

1. **Non-negativity/definiteness:**  
   By inner product axioms, \(\langle x,x\rangle \ge 0\) and equals 0 iff \(x=0\).
   Thus \(\|x\| = \sqrt{\langle x,x\rangle} \ge 0\) and \(\|x\|=0 \iff x=0\).

2. **Homogeneity:**  
   For any scalar \(\alpha\),
   \[
   \|\alpha x\|^2 = \langle \alpha x, \alpha x\rangle
   = \overline{\alpha}\alpha\, \langle x,x\rangle
   = |\alpha|^2 \|x\|^2
   \Rightarrow \|\alpha x\| = |\alpha|\|x\|.
   \]

3. **Triangle inequality:**  
   Using Cauchy–Schwarz,
   \[
   \|x+y\|^2 = \langle x+y, x+y\rangle
   = \|x\|^2 + 2\text{Re}\,\langle x,y\rangle + \|y\|^2
   \le \|x\|^2 + 2|\langle x,y\rangle| + \|y\|^2
   \le \|x\|^2 + 2\|x\|\|y\| + \|y\|^2
   = (\|x\| + \|y\|)^2.
   \]
   Taking square roots gives
   \(\|x+y\| \le \|x\| + \|y\|\).

So \(\|x\| := \sqrt{\langle x,x\rangle}\) is indeed a norm.

---

### (b) Why \(\|x\|_2^2 = \sum_{n=1}^N |x_n|^2\) is NOT a norm

Define \(f(x) = \|x\|_2^2\). This **fails homogeneity**.

Take \(x = (1,0) \in \mathbb{R}^2\) and scalar \(\alpha = 2\):
\[
f(x) = 1^2 + 0^2 = 1,\quad
f(\alpha x) = \|2x\|_2^2 = (2)^2 + 0^2 = 4.
\]
If \(f\) were a norm, we would require
\[
f(\alpha x) = |\alpha| f(x) = 2 \cdot 1 = 2,
\]
but actually \(f(\alpha x) = 4 \neq 2\). Thus homogeneity fails.

Therefore \(f(x) = \|x\|_2^2\) is **not** a valid norm.

---

### (c) Reverse triangle inequality

For any norm, the **reverse triangle inequality**
\[
\big|\|x\| - \|y\|\big| \le \|x - y\|
\]
holds. One standard proof:

Using the triangle inequality on \(x = (x-y) + y\),
\[
\|x\| = \|(x-y) + y\| \le \|x-y\| + \|y\|
\Rightarrow \|x\| - \|y\| \le \|x-y\|.
\]
Swapping the roles of \(x\) and \(y\),
\[
\|y\| - \|x\| \le \|y - x\| = \|x-y\|.
\]
Combining the two inequalities,
\[
-\|x-y\| \le \|x\| - \|y\| \le \|x-y\|
\Rightarrow \big|\|x\| - \|y\|\big| \le \|x-y\|.
\]

## Part 1 – Q4: The Parallelogram Law and Inner Products

### (a) Proving the Parallelogram Law

Assume \(\|\cdot\|\) is induced by an inner product:
\(\|x\|^2 = \langle x,x\rangle\). Then
\[
\begin{aligned}
\|x+y\|^2
&= \langle x+y, x+y\rangle
= \langle x,x\rangle + 2\text{Re}\,\langle x,y\rangle + \langle y,y\rangle,\\
\|x-y\|^2
&= \langle x-y, x-y\rangle
= \langle x,x\rangle - 2\text{Re}\,\langle x,y\rangle + \langle y,y\rangle.
\end{aligned}
\]
Adding,
\[
\|x+y\|^2 + \|x-y\|^2
= 2\langle x,x\rangle + 2\langle y,y\rangle
= 2\|x\|^2 + 2\|y\|^2.
\]
This is exactly the **parallelogram law**.

---

### (b) Why \(\|\cdot\|_\infty\) is not induced by any inner product (for \(N\ge 2\))

In \(\mathbb{R}^N\), the infinity norm is
\(\|x\|_\infty = \max_i |x_i|\). If it were induced by an inner product, it
would satisfy the parallelogram law.

Take \(N=2\), \(x = (1,0)\), \(y = (0,1)\). Then:
\[
\|x\|_\infty = 1,\quad \|y\|_\infty = 1,
\]
\[
x+y = (1,1) \Rightarrow \|x+y\|_\infty = 1,
\]
\[
x-y = (1,-1) \Rightarrow \|x-y\|_\infty = 1.
\]
Compute both sides of the parallelogram law:
\[
\|x+y\|_\infty^2 + \|x-y\|_\infty^2 = 1^2 + 1^2 = 2,
\]
\[
2\|x\|_\infty^2 + 2\|y\|_\infty^2 = 2\cdot 1^2 + 2\cdot 1^2 = 4.
\]
Since \(2 \neq 4\), the parallelogram law fails. Therefore \(\|\cdot\|_\infty\)
cannot come from any inner product.

---

### (c) Weighted inner products \(\langle x,y\rangle_Q = x^\top Q y\)

For \(\langle x,y\rangle_Q = x^\top Qy\) on \(\mathbb{R}^N\) to be a valid inner
product, we need:

1. **Symmetry (or conjugate symmetry):**  
   For real vectors, we need
   \[
   \langle x,y\rangle_Q = x^\top Q y = y^\top Q x = \langle y,x\rangle_Q
   \quad \forall x,y.
   \]
   This holds iff \(Q\) is **symmetric**: \(Q = Q^\top\).

2. **Bilinearity:**  
   Linearity in each argument requires \(Q\) to be fixed and independent of
   \(x,y\). This is automatic once \(Q\) is a fixed matrix.

3. **Positive definiteness:**  
   We need \(\langle x,x\rangle_Q = x^\top Q x > 0\) for all nonzero \(x\),
   and equal to 0 only for \(x=0\). This is exactly the definition of
   **symmetric positive definite** (SPD).

Therefore, \(\langle x,y\rangle_Q = x^\top Q y\) is a valid inner product
if and only if \(Q\) is **symmetric positive definite**.

## Part 1 – Q5: Best Approximation in \(L^2[0,1]\)

We work in the Hilbert space \(S = L^2[0,1]\) with inner product
\(\langle f,g\rangle = \int_0^1 f(t)g(t)\,dt\). We want the best approximation
of
\[
f(t) = t^3
\]
in the 2D subspace
\[
T = \text{span}\{v_1(t), v_2(t)\},\quad v_1(t) = 1,\; v_2(t) = t.
\]
We seek
\[
\hat f(t) = a_1 v_1(t) + a_2 v_2(t) = a_1 + a_2 t.
\]

### (a) Normal equations via orthogonality

The best approximation \(\hat f\) satisfies the orthogonality conditions
\[
\langle f - \hat f, v_1\rangle = 0,\quad \langle f - \hat f, v_2\rangle = 0.
\]

Explicitly:
\[
\int_0^1 \big(t^3 - (a_1 + a_2 t)\big)\cdot 1\, dt = 0,
\]
\[
\int_0^1 \big(t^3 - (a_1 + a_2 t)\big)\cdot t\, dt = 0.
\]

---

### (b) Gram matrix \(G\) and vector \(b\)

We define
\[
G_{ij} = \langle v_j, v_i\rangle,\quad b_i = \langle f, v_i\rangle,
\]
and \(a = (a_1,a_2)^\top\). Then the normal equations can be written as
\[
G a = b.
\]

Compute the entries:

- \(G_{11} = \langle v_1, v_1\rangle = \int_0^1 1\cdot 1\,dt = 1.\)
- \(G_{12} = G_{21} = \langle v_2, v_1\rangle = \int_0^1 t\cdot 1\,dt = \frac{1}{2}.\)
- \(G_{22} = \langle v_2, v_2\rangle = \int_0^1 t^2\,dt = \frac{1}{3}.\)

Thus
\[
G = \begin{bmatrix}
1 & \tfrac{1}{2} \\
\tfrac{1}{2} & \tfrac{1}{3}
\end{bmatrix}.
\]

For \(b\):

- \(b_1 = \langle f, v_1\rangle = \int_0^1 t^3\,dt = \frac{1}{4}.\)
- \(b_2 = \langle f, v_2\rangle = \int_0^1 t^3 \cdot t\,dt = \int_0^1 t^4\,dt = \frac{1}{5}.\)

So
\[
b = \begin{bmatrix} \tfrac{1}{4} \\\ \tfrac{1}{5} \end{bmatrix}.
\]

---

### (c) Solve for \(a_1, a_2\)

We solve \(Ga = b\). The solution is
\[
a_1 = -\frac{1}{5},\qquad a_2 = \frac{9}{10}.
\]
(You can verify this by hand or using a CAS.)

Therefore,
\[
\hat f(t) = -\frac{1}{5} + \frac{9}{10} t.
\]

---

### (d) Error energy \(\|f - \hat f\|^2_{L^2}\)

We compute
\[
\|f - \hat f\|^2
= \int_0^1 \big(t^3 - \hat f(t)\big)^2 dt
= \int_0^1 \left(t^3 + \frac{1}{5} - \frac{9}{10}t\right)^2 dt.
\]

Carrying out the integration (e.g., symbolically), we obtain
\[
\|f - \hat f\|^2 = \frac{9}{700}.
\]

So the best affine approximation in \(L^2[0,1]\) to \(t^3\) is
\(\hat f(t) = -\frac{1}{5} + \frac{9}{10} t\) with squared error \(9/700\).

## Part 1 – Q6: Stability, Computation, and the Dual Basis

### (a) Stability and tiny eigenvalues of the Gram matrix

For a finite basis \(\{v_n\}_{n=1}^N\), with coefficients
\(\alpha = (\alpha_1,\dots,\alpha_N)^\top\) and representation
\(x = \sum_n \alpha_n v_n\), the stability inequality
\[
A \|\alpha\|_2^2 \le \|x\|_2^2 \le B \|\alpha\|_2^2
\]
expresses how nicely the basis behaves. In matrix form,
\(\|x\|_2^2 = \alpha^\top G \alpha\), where \(G\) is the Gram matrix
\(G_{ij} = \langle v_j, v_i\rangle\). The constants \(A,B\) are the **smallest**
and **largest** eigenvalues of \(G\).

If the smallest eigenvalue \(A\) is **tiny**, then there exist coefficient
vectors \(\alpha\) with \(\|\alpha\|_2=1\) such that
\(\|x\|_2^2 = \alpha^\top G \alpha \approx A \ll 1\). In words, we can have
a combination of basis vectors with **large coefficients** that produces a
vector \(x\) of very small norm. This is exactly what happens in the “Nothing
Polynomial” example: the basis functions are almost linearly dependent, so
different large coefficient vectors produce nearly the same function. Numerically,
this makes coefficient recovery extremely unstable: a small perturbation in \(x\)
can cause a large change in \(\alpha\).

---

### (b) Dual basis equals the basis for an orthonormal basis

The **dual basis** \(\{\tilde v_n\}\) is defined by
\[
\langle v_m, \tilde v_n\rangle = \delta_{mn}.
\]
For a general (non-orthonormal) basis there is a nontrivial dual. However,
suppose \(\{v_n\}\) is **orthonormal**. Then
\[
\langle v_m, v_n\rangle = \delta_{mn}.
\]
So the family \(\{v_n\}\) itself satisfies the duality condition. By uniqueness
of the dual basis (for a finite basis), we must have
\[
\tilde v_n = v_n \quad \text{for all }n.
\]

This also shows that the usual coefficient formula for orthonormal bases
\(\hat f = \sum_n \langle f, v_n\rangle v_n\) is equivalent to
\(\hat f = \sum_n \langle f, \tilde v_n\rangle v_n\).

---

### (c) Ill-conditioned Gram matrix and large dual-basis norms

Let \(\{v_n\}\) be a basis with Gram matrix \(G\) and inverse \(H = G^{-1}\).
We have the representation
\[
\tilde v_n = \sum_\ell H_{n\ell} v_\ell.
\]
The eigenvalues of \(H\) are the reciprocals of those of \(G\): if
\(\lambda_{\min}(G) = A\) and \(\lambda_{\max}(G) = B\), then
\[
\lambda_{\min}(H) = \frac{1}{B},\quad \lambda_{\max}(H) = \frac{1}{A}.
\]

If \(G\) is **ill-conditioned**, then \(A\) is very small, so
\(\lambda_{\max}(H) = 1/A\) is very large. Intuitively, this means that some
directions in coefficient space are greatly magnified when we pass through
\(H\). The squared norms of the dual basis vectors are
\(\|\tilde v_n\|^2 = \tilde v_n^\top \tilde v_n\), which can be expressed in
terms of \(H\) and \(G\). A tiny eigenvalue \(A\) implies that some
\(\|\tilde v_n\|\) must be large (on the order of \(1/\sqrt{A}\)).

Therefore, an ill-conditioned Gram matrix forces the dual basis
\(\{\tilde v_n\}\) to contain vectors with very large norms. This is another
manifestation of instability: mapping from a function to its coefficients via
the dual basis amplifies noise dramatically.

## Part 2 – Q1: EVD vs SVD – Foundations and Geometry

### (a) EVD for symmetric matrices

For a real, symmetric matrix \(A \in \mathbb{R}^{N\times N}\), the eigenvalue
decomposition (EVD) is
\[
A = V \Lambda V^\top,
\]
where:
- \(V\) is an **orthogonal** matrix (\(V^\top V = I\)), whose columns are
  eigenvectors of \(A\).
- \(\Lambda\) is a **diagonal** matrix with real eigenvalues \(\lambda_i\) on
  the diagonal.

Geometrically, with respect to the basis given by the columns of \(V\),
the matrix \(A\) simply **scales** each coordinate axis by \(\lambda_i\). That is,
if \(x = V c\), then
\[
Ax = V \Lambda c,
\]
so in the eigenbasis, \(A\) stretches or compresses along each eigenvector
direction independently.

---

### (b) SVD for general matrices – rotation, scaling, rotation

For any real matrix \(A \in \mathbb{R}^{M\times N}\), the singular value
decomposition (SVD) is
\[
A = U \Sigma V^\top,
\]
where:
- \(U \in \mathbb{R}^{M\times M}\) is orthogonal: \(U^\top U = I_M\).
- \(V \in \mathbb{R}^{N\times N}\) is orthogonal: \(V^\top V = I_N\).
- \(\Sigma \in \mathbb{R}^{M\times N}\) is diagonal (possibly rectangular) with
  nonnegative entries \(\sigma_1 \ge \cdots \ge \sigma_R > 0\) on the diagonal.

Geometrically, applying \(A\) to a vector \(x\) can be viewed as:
1. **Right rotation**: compute \(V^\top x\), i.e., express \(x\) in the
   orthonormal basis of right singular vectors.
2. **Scaling**: apply \(\Sigma\), which scales each coordinate by \(\sigma_i\).
3. **Left rotation**: apply \(U\), which rotates the result into the output
   space.

The unit ball in \(\mathbb{R}^N\) is thus mapped by \(A\) to a hyper-ellipse
whose principal axes are the columns of \(U\) with lengths \(\sigma_i\).

---

### (c) Relationship between SVD and the EVD of \(A^\top A\) and \(AA^\top\)

Given \(A = U\Sigma V^\top\), we have
\[
A^\top A = V \Sigma^\top U^\top U \Sigma V^\top
         = V \Sigma^2 V^\top,
\]
so the **right singular vectors** (columns of \(V\)) are eigenvectors of
\(A^\top A\), and the eigenvalues are \(\sigma_i^2\).

Similarly,
\[
AA^\top = U \Sigma V^\top V \Sigma^\top U^\top
        = U \Sigma^2 U^\top,
\]
so the **left singular vectors** (columns of \(U\)) are eigenvectors of
\(AA^\top\), with the same eigenvalues \(\sigma_i^2\).

---

### (d) When SVD = EVD

If \(A\) is **symmetric** and **positive semidefinite**, then the EVD
\(A = V\Lambda V^\top\) has \(\Lambda \ge 0\). In that case, we can take
\(\Sigma = \Lambda^{1/2}\) and set \(U = V\), so the EVD and SVD coincide.

More generally, for a real symmetric matrix \(A\) with possibly negative
eigenvalues, the singular values of \(A\) are \(|\lambda_i|\), and the singular
vectors equal the eigenvectors. The EVD and SVD differ only by the signs of
the eigenvalues.

## Part 2 – Q2: Pseudoinverse and Minimum-Norm Least Squares

### (a) General SVD formula for \(A^\dagger\)

Let \(A \in \mathbb{R}^{M\times N}\) have rank \(R\), with SVD
\[
A = U \Sigma V^\top,
\]
where \(\Sigma\) has positive singular values \(\sigma_1,\dots,\sigma_R\) and
zeros elsewhere. Then the **Moore–Penrose pseudoinverse** of \(A\) is
\[
A^\dagger = V \Sigma^\dagger U^\top,
\]
where \(\Sigma^\dagger\) is obtained by:
- Taking reciprocals of the nonzero singular values:
  \(1/\sigma_1,\dots,1/\sigma_R\),
- Transposing the rectangular shape.

In coordinates:
\[
A^\dagger = \sum_{r=1}^R \frac{1}{\sigma_r} v_r u_r^\top,
\]
where \(u_r, v_r\) are the left/right singular vectors.

---

### (b) Three optimality properties of \(x^\dagger = A^\dagger y\)

The solution \(x^\dagger = A^\dagger y\) is “optimal” in three standard senses:

1. **Least-squares (LS) minimizer:**
   \[
   x^\dagger = \arg\min_x \|Ax - y\|_2.
   \]
   It gives the minimum residual norm.

2. **Minimum-norm among LS solutions:**
   If the LS problem has infinitely many solutions (e.g., rank-deficient \(A\)),
   then \(x^\dagger\) is the unique solution of **smallest Euclidean norm**:
   \[
   x^\dagger = \arg\min\{\|x\|_2 \;:\; \|Ax - y\|_2 = \min\}.
   \]

3. **Orthogonality of residual:**
   The residual
   \[
   r^\dagger = y - A x^\dagger
   \]
   is orthogonal to the column space of \(A\), i.e.,
   \[
   A^\top r^\dagger = 0.
   \]
   This characterizes \(x^\dagger\) as the orthogonal projection of \(y\) onto
   the column space of \(A\).

---

### (c) Tall full-column-rank case: \(A^\dagger = (A^\top A)^{-1}A^\top\)

Assume \(A \in \mathbb{R}^{M\times N}\) with full column rank \(R = N < M\).
Then \(\Sigma\) is \(N\times N\) and **invertible**, so
\[
A = U \Sigma V^\top,\quad
A^\dagger = V \Sigma^{-1} U^\top.
\]

Now,
\[
A^\top A = V \Sigma U^\top U \Sigma V^\top
         = V \Sigma^2 V^\top.
\]
Hence
\[
(A^\top A)^{-1} = V \Sigma^{-2} V^\top.
\]
Multiplying by \(A^\top\),
\[
(A^\top A)^{-1} A^\top
= V \Sigma^{-2} V^\top A^\top
= V \Sigma^{-2} V^\top V \Sigma U^\top
= V \Sigma^{-1} U^\top
= A^\dagger.
\]

Thus the SVD formula \(A^\dagger = V\Sigma^{-1}U^\top\) reduces to the familiar
\[
A^\dagger = (A^\top A)^{-1} A^\top
\]
for tall full-rank matrices.


## Part 2 – Q3: Instability and Regularization

### (a) Instability of LS: reconstruction error and small singular values

Let \(y = A x^\star + e\), where \(e\) is noise (e.g. white noise with
\(\mathbb{E}[e] = 0\), \(\mathbb{E}[ee^\top] = \sigma^2 I\)).
The least-squares solution is
\[
\hat x_{\text{LS}} = A^\dagger y = A^\dagger(Ax^\star + e)
= x^\star + A^\dagger e.
\]
So the error is
\[
\hat x_{\text{LS}} - x^\star = A^\dagger e.
\]

Using the SVD \(A = U\Sigma V^\top\), we write
\[
A^\dagger = V \Sigma^\dagger U^\top.
\]
Let \(e' = U^\top e\), so that the components of \(e\) in the left singular
basis are \(e'_r\). Then
\[
\hat x_{\text{LS}} - x^\star
= V \Sigma^\dagger e'
= \sum_{r=1}^R \frac{1}{\sigma_r} e'_r\, v_r.
\]
Assuming the noise is white with variance \(\sigma^2\),
\(\mathbb{E}[(e'_r)^2] = \sigma^2\), and the expected squared error is
\[
\mathbb{E}\big[\|\hat x_{\text{LS}} - x^\star\|_2^2\big]
= \mathbb{E}\left[\sum_{r=1}^R \frac{(e'_r)^2}{\sigma_r^2}\right]
= \sigma^2 \sum_{r=1}^R \frac{1}{\sigma_r^2}.
\]

If some singular values \(\sigma_r\) are **very small**, their reciprocals
\(1/\sigma_r\) are huge, and thus even modest noise \(e\) gets amplified,
producing large reconstruction error. This is why small singular values are
“dangerous” for LS in ill-conditioned problems.

---

### (b) Tikhonov regularization in the SVD basis

Tikhonov regularization solves
\[
\hat x_\lambda = \arg\min_x \big(\|Ax - y\|_2^2 + \lambda\|x\|_2^2\big),
\]
with solution
\[
\hat x_\lambda = (A^\top A + \lambda I)^{-1} A^\top y.
\]
Using the SVD \(A = U\Sigma V^\top\),
\[
A^\top A = V\Sigma^2 V^\top,
\]
so
\[
A^\top A + \lambda I = V(\Sigma^2 + \lambda I)V^\top
\Rightarrow (A^\top A + \lambda I)^{-1} =
V(\Sigma^2 + \lambda I)^{-1} V^\top.
\]
Then
\[
\hat x_\lambda
= V(\Sigma^2 + \lambda I)^{-1} \Sigma U^\top y.
\]

If we expand in the singular vector basis, with \(y' = U^\top y\), we get
\[
\hat x_\lambda
= \sum_{r=1}^R \frac{\sigma_r}{\sigma_r^2 + \lambda}\, y'_r\, v_r.
\]

Comparing with LS (which uses multiplier \(1/\sigma_r\) on \(y'_r\)), we see that
Tikhonov introduces a **damped multiplier**
\[
\phi_r(\lambda) = \frac{\sigma_r}{\sigma_r^2 + \lambda}
\]
for each singular direction.

This yields a decomposition of the error into:
- A **bias** term coming from shrinking the true component \(Ax^\star\)
  (multipliers are less than \(1/\sigma_r\)),
- A **variance** term from noise, which is reduced when
  \(\sigma_r \ll \sqrt{\lambda}\) because \(\phi_r(\lambda)\) is small.

---

### (c) Truncated SVD vs Tikhonov

- **Truncated SVD (TSVD):**  
  We keep only the largest \(R'\) singular values and set the rest to zero. The
  effective multiplier for LS in direction \(r\) is
  \[
  m_r^{\text{TSVD}} = 
  \begin{cases}
    1/\sigma_r, & r \le R',\\
    0, & r > R'.
  \end{cases}
  \]
  Thus, all contributions from singular values beyond the cutoff \(R'\) are
  **discarded** entirely. These neglected directions contribute to the
  **approximation bias**.

- **Tikhonov:**  
  Uses a smooth damping multiplier
  \[
  m_r^{\text{Tik}} = \frac{\sigma_r}{\sigma_r^2 + \lambda}.
  \]
  For large \(\sigma_r\), this is approximately \(1/\sigma_r\) (weak damping),
  while for small \(\sigma_r\), the multiplier behaves like \(\sigma_r/\lambda\)
  and hence heavily suppresses noise amplification.

**Comparison:**

- TSVD applies a **hard threshold**: singular components below the cutoff are
  dropped (multiplier 0), above the cutoff they are treated as in LS.
- Tikhonov applies a **soft, continuous damping**: every component is retained,
  but small singular values are scaled down strongly.

In both cases, the **small singular values** (the ill-conditioned directions)
are the ones contributing most to instability, and both regularization methods
aim to control their effect: TSVD by discarding them, Tikhonov by shrinking
them.


## Part 2 – Q4: Total Least Squares (TLS)

### (a) LS vs TLS error models

Ordinary Least Squares (LS) assumes that the **design matrix** \(A\) is exact
and all errors are in the **output** \(y\). The model is
\[
y = A x^\star + e,
\]
and LS finds \(x\) to minimize \(\|Ax - y\|_2\).

Total Least Squares (TLS) relaxes this assumption by allowing **errors in both**
\(A\) and \(y\). We write
\[
(A + \Delta A)x = y + \Delta y,
\]
and seek \((\Delta A, \Delta y, x)\) that jointly minimize
\(\|[\Delta A\ \Delta y]\|_F\), subject to the existence of a solution for the
perturbed system. This is more realistic when both inputs and outputs are noisy.

---

### (b) Why the TLS solution comes from the smallest singular vector of \(C=[A\;|\;y]\)

Let \(C = [A\;|\;y] \in \mathbb{R}^{M\times (N+1)}\). TLS looks for the smallest
perturbation \(\Delta C\) such that \((A+\Delta A)x = y + \Delta y\) is
**exactly consistent**:
\[
\exists\, x \quad \text{s.t.} \quad (C + \Delta C)\begin{bmatrix} x \\ -1 \end{bmatrix} = 0.
\]

Equivalently, we seek a rank-\(N\) approximation \(\tilde C\) of \(C\) that is
closest in Frobenius norm:
\[
\tilde C = \arg\min_{\operatorname{rank}(\tilde C) \le N} \|C - \tilde C\|_F.
\]

By the Eckart–Young–Mirsky theorem, the best rank-\(N\) approximation of \(C\)
is obtained by truncating its SVD:
\[
C = \tilde U \tilde \Sigma \tilde V^\top,\quad
\tilde C = \tilde U \tilde \Sigma_N \tilde V_N^\top,
\]
where \(\tilde \Sigma_N\) zeros out the smallest singular value \(\gamma_{N+1}\).
The right singular vector corresponding to \(\gamma_{N+1}\) (the **smallest**
singular value) is \(\tilde v_{N+1}\).

We can partition this vector as
\[
\tilde v_{N+1} = \begin{bmatrix} z \\ -1 \end{bmatrix},
\]
up to an overall scale. The condition
\[
C \tilde v_{N+1} \approx 0
\]
implies that the closest rank-\(N\) system has an **exact solution** \(x = z\).

Thus, the TLS solution \(x_{\text{TLS}}\) is given (up to normalization) by
\(-v_1 / v_{N+1}\) or equivalently, by interpreting the last right singular
vector of \(C\) as \([x_{\text{TLS}}; -1]\). This connects the TLS solution
directly to the **smallest singular value and its right singular vector**.

## Part 2 – Q5: PCA and Low-Rank Approximation

### (a) The “grand connection” – what matrices are approximated?

- **PCA:**  
  In PCA, we typically center the data matrix \(\tilde X \in \mathbb{R}^{M\times N}\)
  (rows = samples, columns = features). We then find the best rank-\(K\)
  approximation of this centered data:
  \[
  \tilde X \approx \tilde X_K = U_K \Sigma_K V_K^\top.
  \]
  Geometrically, PCA finds the rank-\(K\) matrix that is closest to \(\tilde X\)
  in Frobenius norm, which corresponds to projecting data onto the subspace
  spanned by the top \(K\) singular vectors.

- **TLS:**  
  In TLS, we form the augmented matrix
  \[
  C = [A\;|\;y] \in \mathbb{R}^{M\times (N+1)}.
  \]
  TLS finds the **best rank-\(N\)** approximation of \(C\) in Frobenius norm.
  Thus, it is also a low-rank approximation problem, but the matrix is the
  *joint* data \((A, y)\) rather than just features \(\tilde X\).

Both problems are instances of “best low-rank approximation via SVD”.

---

### (b) PCA: SVD vs covariance method

Let \(\tilde X\) be the centered data matrix. Its SVD is
\[
\tilde X = U\Sigma V^\top.
\]
The sample covariance matrix (for data arranged as rows) is
\[
S = \frac{1}{N} \tilde X^\top \tilde X.
\]
Substituting the SVD:
\[
S = \frac{1}{N} V \Sigma^\top U^\top U \Sigma V^\top
  = V \left(\frac{1}{N}\Sigma^2\right) V^\top.
\]
So:
- The **eigenvectors** of \(S\) are the **right singular vectors** (columns of
  \(V\)).
- The **eigenvalues** of \(S\) are \(\sigma_r^2 / N\).

If instead we center data by subtracting the mean from each row and treat
columns as variables, the covariance matrix becomes
\[
S' = \frac{1}{N} \tilde X \tilde X^\top = U \left(\frac{1}{N}\Sigma^2\right) U^\top,
\]
whose eigenvectors are the **left singular vectors** \(U\).

Thus, **performing eigendecomposition of the covariance matrix is equivalent
to performing SVD of the centered data matrix**: they yield the same principal
directions and variances, just expressed in different spaces (feature space
vs sample space).

## Part 2 – Q6: The Grammian Operator and Riesz Bases

### (a) Gram matrix and linear independence

Given a finite set of vectors \(\{v_1,\dots,v_N\}\) in an inner product space,
the Gram matrix is
\[
G_{nm} = \langle v_n, v_m\rangle.
\]

- If the vectors are **linearly dependent**, there exists a nonzero vector
  \(\alpha = (\alpha_1,\dots,\alpha_N)^\top\) such that
  \(\sum_n \alpha_n v_n = 0\). Then
  \[
  \alpha^\top G \alpha
  = \left\langle \sum_n \alpha_n v_n, \sum_m \alpha_m v_m\right\rangle
  = \langle 0,0\rangle
  = 0.
  \]
  So \(G\) is **not** positive definite and thus not invertible.

- If the vectors are **linearly independent**, and \(\alpha \neq 0\), then
  \(\sum_n \alpha_n v_n \neq 0\), so
  \[
  \alpha^\top G \alpha = \left\|\sum_n \alpha_n v_n\right\|^2 > 0.
  \]
  Hence \(G\) is **positive definite** and therefore invertible.

Thus, \(G\) is invertible if and only if \(\{v_n\}\) is linearly independent.

---

### (b) Riesz bases, Gram eigenvalues, and “well-conditioned” bases

An infinite sequence \(\{v_n\}_{n=1}^\infty\) in a Hilbert space is a
**Riesz basis** if there exist constants \(A,B > 0\) such that for all finite
coefficient sequences \(\alpha = (\alpha_1,\dots,\alpha_N)\),
\[
A\|\alpha\|_2^2 \;\le\; \left\|\sum_{n=1}^N \alpha_n v_n\right\|_2^2
\;\le\; B\|\alpha\|_2^2.
\]
In finite dimensions this is precisely the stability inequality from Part 1,
with \(A,B\) the **extreme eigenvalues** of the Gram matrix \(G\).

- If \(A\) and \(B\) are reasonably close (i.e. \(B/A\) is not huge), the basis
  is **well-conditioned**: the mapping from coefficients to vectors and back is
  stable; small changes in coefficients produce proportionally small changes
  in the represented vector, and vice versa.

- For an orthonormal basis, \(G = I\) so \(A=B=1\); this is the “ideal” case
  of perfect conditioning.

A Riesz basis is therefore a **well-conditioned generalization of an
orthonormal basis**: it may not be orthogonal, but its Gram operator has
eigenvalues uniformly bounded away from 0 and infinity, ensuring stability of
representation and reconstruction.

In [10]:
# Q13 – Gram Matrix, LS, and Stability via Truncated SVD
results_q13 = run_monomial_experiment()
results_q13


[23:10:38] [INFO] --- (a) LS Solution ---
[23:10:38] [INFO] a_LS = [1.01226382 0.85222614 0.83977259]
[23:10:38] [INFO] Condition number κ(G) = 4.86e+02
[23:10:38] [INFO] --- (b) TSVD Solution (R'=2) ---
[23:10:38] [INFO] Singular values of A: [8.40549389 2.51909728 0.38125914]
[23:10:38] [INFO] a_TSVD = [1.00012127 0.92119543 0.77331851]
[23:10:38] [INFO] --- (c) Coefficient Norms ---
[23:10:38] [INFO] ||a_LS||_2   = 1.5672
[23:10:38] [INFO] ||a_TSVD||_2 = 1.5642
[23:10:38] [INFO] Saved figure to outputs/figures/monomial_ls_tsvd.png
[23:10:38] [INFO] --- (d) Residual Errors ---
[23:10:38] [INFO] LS residual      = 0.039550
[23:10:38] [INFO] TSVD residual    = 0.054028


{'a_LS': array([1.01226382, 0.85222614, 0.83977259]),
 'a_TSVD': array([1.00012127, 0.92119543, 0.77331851]),
 'res_LS': np.float64(0.03955014701713053),
 'res_TSVD': np.float64(0.054027830403766464),
 'kappa_G': np.float64(486.0555972720651),
 'R_vals': [1, 2, 3],
 'residuals': [np.float64(1.0954779693372632),
  np.float64(0.054027830403766464),
  np.float64(0.03955014701713047)],
 'coeff_norms': [np.float64(1.5027354409172708),
  np.float64(1.5642458541997155),
  np.float64(1.5672222056914193)]}

### Q13 – Discussion

The Gram matrix \(G = A^\top A\) for the monomial basis \(\{1, t, t^2\}\) on
\([0,1]\) has a **large condition number** \(\kappa(G)\), reflecting the fact
that monomials are poorly conditioned as a basis (highly correlated columns).

- The ordinary LS solution \(a_{\text{LS}}\) exactly inverts the small
  singular values, so the coefficients can become large and sensitive to noise.
- The truncated SVD solution \(a_{\text{TSVD}}\) with \(R'=2\) “drops” the
  smallest singular direction, yielding a smaller coefficient norm and improved
  numerical stability, at the cost of a slightly larger residual.

The residual vs. truncation level \(R'\) reveals the **bias–variance trade-off**:
- Small \(R'\): higher bias (poorer approximation), but more stable.
- Larger \(R'\): lower bias but increased sensitivity to ill-conditioning.

In practice, one chooses \(R'\) so that most of the energy of the singular
values is retained, while discarding the noise-dominated directions.


In [11]:
# Q14 – When does TLS help?
results_q14 = run_tls_experiments()
results_q14

[23:11:03] [INFO] ==================================================
[23:11:03] [INFO] EXPERIMENT 1: Noise only in y (x is clean)
[23:11:04] [INFO] ==================================================
[23:11:04] [INFO] True theta:   2.5000
[23:11:04] [INFO] OLS estimate: 2.4455 (error = 0.0545)
[23:11:04] [INFO] TLS estimate: 2.4567 (error = 0.0433)
[23:11:04] [INFO] ==================================================
[23:11:04] [INFO] EXPERIMENT 2: Noise in both x and y
[23:11:04] [INFO] ==================================================
[23:11:04] [INFO] True theta:   2.5000
[23:11:04] [INFO] OLS estimate: 2.4226 (error = 0.0774)
[23:11:04] [INFO] TLS estimate: 2.5105 (error = 0.0105)
[23:11:04] [INFO] Saved figure to outputs/figures/tls_experiments.png


{'theta_true': 2.5,
 'exp1': (2.4455063556930843,
  2.4567233922945806,
  0.05449364430691572,
  0.04327660770541941),
 'exp2': (2.4226328044936127,
  2.510535293503578,
  0.07736719550638727,
  0.010535293503577847)}

### Q14 – Discussion: When Does TLS Help?

In **Experiment 1**, only the output \(y\) is corrupted by noise, while the
input \(x\) is exact. LS is designed exactly for this model and tends to
perform better than TLS. TLS may over-correct by trying to adjust both
\(x\) and \(y\), even though the true \(x\) is noise-free, leading to a larger
parameter error \(|\hat\theta_{\text{TLS}} - \theta_{\text{true}}|\).

In **Experiment 2**, both \(x\) and \(y\) are corrupted with noise of similar
magnitude. The LS model is now mismatched (it still assumes exact \(x\)), so
it typically underestimates the uncertainty and can be biased. TLS, which
explicitly allows perturbations in both \(A\) (inputs) and \(y\), is more
appropriate and often yields a smaller parameter error.

In summary:
- LS is optimal (in a least-squares sense) when the design matrix is exact and
  noise is only in \(y\).
- TLS becomes advantageous when measurement errors affect **both** inputs and
  outputs, matching the TLS error model.

In [12]:
# Q15 – PCA and Low-Rank Approximation of a Digit Image
results_q15 = run_pca_steps()
results_q15


[23:11:29] [INFO] ============================================================
[23:11:29] [INFO] PCA Step-by-Step: Geometry, Decorrelation, Compression
[23:11:29] [INFO] ============================================================
[23:11:29] [INFO] Image shape: (16, 16), Total values: 256
[23:11:29] [INFO] ---- (a) Geometric Transformation ----
[23:11:29] [INFO] First principal component v1: [ 0.00000000e+00 -2.22044605e-16  0.00000000e+00  6.93889390e-18
  3.84609246e-01  3.84609246e-01  4.25722734e-01  4.25722734e-01
  4.63018339e-01  3.56771329e-01  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
[23:11:29] [INFO] Saved PCA step figure to outputs/figures/pca_steps.png
[23:11:29] [INFO] ---- (b) Statistical Decorrelation ----
[23:11:29] [INFO] Off-diagonal energy: Original = 30695.07, PCA = 0.00e+00
[23:11:30] [INFO] Saved covariance heatmaps to outputs/figures/pca_covariance_heatmaps.png
[23:11:30] [INFO] ---- (c) Dimensionality Reduc

{'errors_svd': {1: np.float64(603.0686128922605),
  3: np.float64(154.6542262870419),
  5: np.float64(6.720030319178291e-13),
  10: np.float64(6.73027117740753e-13),
  16: np.float64(6.73027117740753e-13)},
 'error_cov': np.float64(154.65422628704192),
 'diff': np.float64(9.86043970341699e-13),
 'off_orig': np.float64(30695.072894521687),
 'off_pca': np.float64(0.0)}

### Q15 – Discussion: PCA, Decorrelation, and Compression

The SVD-based PCA of the \(16\times16\) digit image shows:

- **Geometric transformation:**  
  Rotating the data into the principal component basis (matrix \(Z\)) aligns the
  coordinate axes with directions of maximum variance. The first principal
  component vector \(v_1\) often corresponds to the dominant stroke pattern of
  the digit.

- **Statistical decorrelation:**  
  The covariance of the original centered data \(C_{\text{orig}}\) has
  significant off-diagonal energy, indicating correlated columns (pixels).
  After rotation, the covariance \(C_{\text{pca}}\) becomes nearly diagonal:
  off-diagonal energy is greatly reduced. PCA thus decorrelates the data.

- **Compression:**  
  Using only the top \(k=3\) principal components gives a reconstruction with
  relatively small Frobenius error \(\|X - \hat X\|_F\) compared to the original
  energy. The storage cost drops from 256 numbers to \(17k+16=67\), giving a
  substantial compression ratio while preserving the main structure of the digit.

- **Covariance vs. SVD method:**  
  Implementing PCA via eigendecomposition of the covariance matrix
  \(C = \frac{1}{n}\tilde{X}^\top\tilde{X}\) yields essentially the same
  reconstructed image as the direct SVD method (for \(k=3\)), and the numerical
  difference is on the order of machine precision. This confirms that the two
  formulations are theoretically equivalent.


In [13]:
# Q18 – Regression with Explicit and Kernel Methods
results_q18 = run_regression_comparison()
results_q18

[23:11:55] [INFO] ============================================================
[23:11:55] [INFO] REGRESSION COMPARISON: Explicit vs. Kernel Methods
[23:11:55] [INFO] ============================================================
[23:11:55] [INFO] True function: y = 2*sin(2x) + 0.5*x^3
[23:11:55] [INFO] Samples: 50, Noise std: 0.5
[23:11:55] [INFO] ============================================================
[23:11:55] [INFO] MEAN SQUARED ERROR (MSE) vs. TRUE FUNCTION
[23:11:55] [INFO] ============================================================
[23:11:55] [INFO] Method                    MSE         
[23:11:55] [INFO] ------------------------------------------------------------
[23:11:55] [INFO] Linear OLS                3.6367e+00  
[23:11:55] [INFO] Gramian (Linear)          3.6372e+00  
[23:11:55] [INFO] Polynomial (deg 5)        1.8155e-01  
[23:11:55] [INFO] RBF Kernel                2.8419e-01  
[23:11:55] [INFO] Best method: Polynomial (deg 5) (MSE = 1.8155e-01)
[23:11:55] [INFO] 

{'mse_ols': 3.6367461199253,
 'mse_poly': 0.18155439686612188,
 'mse_gram_lin': 3.6371733720051616,
 'mse_rbf': 0.2841922590985657,
 'best_method': 'Polynomial (deg 5)'}

### Q18 – Discussion: Explicit Features vs Kernel Methods

We compare:
- **Linear OLS:** features \(\phi_{\text{lin}}(x) = [1,x]\),
- **Gramian Ridge with linear features:** dual formulation using
  \(K = \Phi_{\text{lin}} \Phi_{\text{lin}}^\top\),
- **Polynomial regression (degree 5):** features \(\phi_{\text{poly}}(x) =
  [1,x,\dots,x^5]\),
- **RBF kernel ridge regression:** kernel
  \(k(x_i,x_j) = \exp(-\gamma\|x_i - x_j\|^2)\).

**Why OLS and Gramian-linear give identical results:**  
The Gramian ridge method with linear features solves in the dual space using
\(\alpha\) and the Gram matrix \(K\), but the resulting predictions
\(\hat y = K\alpha\) correspond exactly to a linear model in the original
feature space. Algebraically, one can show that the primal solution
\(w = (\Phi^\top\Phi + \lambda I)^{-1}\Phi^\top y\) and the dual solution
\(\alpha = (K + \lambda I)^{-1} y\) are equivalent, and they produce identical
\(\hat y\). Thus, Gramian-linear is simply a reparameterization of linear OLS
(with regularization) and yields the same fit.

**Why the RBF kernel outperforms polynomial regression:**  
The ground-truth function \(y(x) = 2\sin(2x) + 0.5x^3\) is highly nonlinear and
oscillatory. A fixed-degree polynomial (e.g., degree 5) can approximate it
only roughly over the interval, and may oscillate or extrapolate poorly. The
RBF kernel, on the other hand, implicitly maps data into an infinite-dimensional
feature space of localized bumps and can flexibly adapt to both the sinusoidal
and cubic components. This extra flexibility leads to a lower MSE against the
true function.

**When the Gram matrix approach is essential:**  
For kernels like the RBF, the feature space is infinite-dimensional and we
cannot explicitly construct \(\phi(x)\). The Gram matrix approach (kernel trick)
allows us to work entirely in terms of inner products
\(k(x_i,x_j) = \langle\phi(x_i),\phi(x_j)\rangle\) without ever forming \(\phi\).
In such cases, the dual/Gram formulation is not just an alternative; it is
the **only practical way** to implement the regression.